# 18 · Convergence with the sampling of the skewers (TNG50-2, the `points` sweep)


How finely must the points along a skewer be spaced? SkewerJAX's default route on straight skewers needs no sampling:
it deposits every Voronoi cell on its **exact crossing** with the skewer and integrates the line profile over each
crossing. A skewer known only by its points (any path, curved ones included) is cut into the **dual segments** of the
points, the cells crossed found between the midpoints of consecutive points (`pieces="segments"`). This notebook
measures what that costs in accuracy: five spacings $dx$ = 1, 0.5, 0.1, 0.05, 0.01 cMpc, each with the integrated
profile and the exact frame (its $a(s)$ table on the same points), against the fiducial run (the exact crossings,
$a(s)$ every 0.05 cMpc) and against the finest spacing. TNG50-2 snapshot 25 ($z = 3$), $128 \times 128$ skewers along $y$,
1 km/s pixels, $\mathrm{Ly}\alpha$ and Si III 1206 (`run_production.py --sweep points`).


Every run is a folder of `scripts/production/run_production.py` (`<OUT>/<SWEEP>/<run>/run.json`) that names its
lattice in the campaign's skewer store; `load_run` reads it. The statistics follow notebooks 11 and 18 on **one line of
sight** ($y$): the transmission $T = e^{-\tau}$ of every line and of the total, its mean $\bar T$, the overflux
$\delta = T/\bar T - 1$, P1D on its native modes and $P(k, \mu)$ of the total in four wedges
(`summary_stats.run_statistics`). One line of sight gives no ensemble, so the errors come from a **delete-one-block
jackknife**: the lattice is cut into $4 \times 4$ squares of skewers and each sample leaves one square out (the
3D power of the field with that square set to zero, divided by the fraction kept). Every run is compared with the
reference **bin by bin**, as the relative difference $X_{\rm run}/X_{\rm ref} - 1$, with the jackknife error of that
ratio (`compare_runs`): the runs share the skewer positions and the blocks, so the sample variance they share cancels.
The last part gives, per run, **the largest $k$ up to which every bin differs by less than 0.1%, 0.5% and 1%**
(`k_max_table`; P1D on every native mode, $P(k, \mu)$ on every $|k|$ bin).

Colours: the fiducial run of the campaign in black in every notebook; the other runs from light (coarse) to dark (fine)
on one scale shared by notebooks 18–21.


- **Part A — the stored skewers read back:** every run's lattice, pixels, sampling and dataset, and the checks of the
  pairing (the same positions and pixels in every run).
- **Part B — the statistics of every run** and their comparison with the reference.
- **Part C — the mean transmission.**
- **Part D — P1D** of every line and of the total.
- **Part E — the wedges of $P(k, \mu)$** of the total.
- **Part F — the converged range:** the largest $k$ within 0.1%, 0.5% and 1% of the reference, and what it implies.

In [ ]:
import os
from pathlib import Path

# ---- configuration: the sweep (scripts/production/run_production.py --sweep points)
SWEEP = "points"
RUNS = ["dx_1cMpc", "dx_0p5cMpc", "dx_0p1cMpc", "dx_0p05cMpc", "dx_0p01cMpc", "reference"]   # coarse to fine, then the fiducial
REFERENCE = "reference"                           # the fiducial run (exact crossings): the converged reference
FINEST = "dx_0p01cMpc"                            # the second reference of Part F
FIDUCIAL = "reference"                            # black in every figure
ORDER = RUNS                                      # the colour scale, coarse to fine
DX = {"dx_1cMpc": 1.0, "dx_0p5cMpc": 0.5, "dx_0p1cMpc": 0.1, "dx_0p05cMpc": 0.05, "dx_0p01cMpc": 0.01}   # [cMpc]
RUN_LABELS = {**{r: f"dx = {x:g} cMpc" for r, x in DX.items()}, "reference": "fiducial (crossings)"}
TITLE_STEM = "Sampling points"
LINTHRESH = 1e-4                                  # [%]: the residual axes are linear within it
OUT = Path(os.environ.get("SKEWERJAX_PRODUCTION", Path.home() / "skewerjax_production"))   # the --out of the campaign
N_SUB = None                                      # None: each run's lattice; 32: a quick check on the nested 32^2 sub-lattice
N_BLOCKS = 4                                      # jackknife: N_BLOCKS x N_BLOCKS squares of skewers
MU_EDGES = (0.0, 0.5, 0.8, 0.95, 1.0)             # the four wedges of P(k, mu) (notebooks 10 and 11's wedges of xi)
TOLS = (0.001, 0.005, 0.01)                       # the convergence levels of Part F
X_OF_RUN, X_LABEL, X_SCALE, X_TICKS = DX, "sampling spacing $dx$ [cMpc]", "log", None

# ---- imports
os.environ.setdefault("JAX_PLATFORMS", "cpu")
import logging; logging.getLogger("jax._src.xla_bridge").setLevel(logging.CRITICAL)
import json, sys, time
import numpy as np
import matplotlib.pyplot as plt
from skewerjax.utils.plotting import use_style, use_backend
# use_backend()                                    # interactive figures in a live notebook
use_style()
from skewerjax.utils.plotting import (add_panel_letters, field_names, run_styles, plot_runs_mean_flux, plot_runs_p1d,
                                      plot_runs_wedges, plot_k_max_summary)
from skewerjax.summary_stats import (LatticeWedges, block_labels, compare_runs, format_k_max_table, k_max_table,
                                     rebin_pixels, run_statistics, set_k_edges)
from skewerjax.utils.reference_data import P1D_DESI_DR1, nearest_reference
from skewerjax.utils.paths import PRODUCTION_DIR, PRODUCTION_SCRIPTS_DIR   # the run folders; the reader of the skewer store
sys.path.insert(0, str(PRODUCTION_SCRIPTS_DIR))
from skewer_store import load_run
from skewerjax.utils.plotting import PALETTE

## Part A — The stored skewers read back

- `OUT`, `SWEEP`, `RUNS`: the run folders `<OUT>/<SWEEP>/<run>` (default `$SKEWERJAX_PRODUCTION`, or
  `~/skewerjax_production`); `REFERENCE` the fiducial run, `FINEST` the finest spacing.
- `N_SUB`: `None` reads each run's $128 \times 128$ lattice; 32 reads its nested $32 \times 32$ sub-lattice (a quick check).
- `N_BLOCKS`, `MU_EDGES`, `TOLS`: the jackknife squares, the wedges, the convergence levels.

The table lists, per run, its lattice, pixels, sampling (in cMpc and in km/s of Hubble flow), route (`streaming`: the
crossings; `sampled`: the dual segments), dataset and cost. The checks: every run has the same skewers (positions) and
the same pixels, so the comparisons are skewer by skewer and pixel by pixel.

In [ ]:
RECS = {r: json.loads((OUT / SWEEP / r / "run.json").read_text()) for r in RUNS}
S = {r: load_run(OUT / SWEEP / r, N_SUB) for r in RUNS}          # tau (n_lines, N_sk, N_pix), positions, columns, v, dv, L, u
REF = S[REFERENCE]
L, U, PERIOD = REF.L, REF.u, REF.period                          # [cMpc], aH [km/s per cMpc], v_per [km/s]
N = REF.n_per_axis
Z, A_SNAP = RECS[REFERENCE]["snapshot"]["redshift"], RECS[REFERENCE]["snapshot"]["a"]
LINE_IDS = REF.line_ids
FIELDS = (*LINE_IDS, "total")
NAMES = field_names(FIELDS)
print(f"{'run':<13} {'sim':<8} {'lattice':>8} {'N_pix':>6} {'dv [km/s]':>10} {'dx [cMpc]':>10} {'dx [km/s]':>10} {'route':<10} "
      f"{'dataset':<11} {'computed':>8} {'wall':>8}")
for r in RUNS:
    rec, s, sm = RECS[r], S[r], RECS[r]["sampling"]
    print(f"{r:<13} {s.sim:<8} {s.n_per_axis:>5}^2 {s.v.size:>6} {s.dv:>10.4f} {sm['points_cMpc']:>10.3g} {sm['points_kms']:>10.3g} "
          f"{sm['route']:<10} {rec['dataset']['hash']:<11} {rec['timings']['computed']:>8} {rec['wall_s']:>7.0f}s")
    assert np.isfinite(s.tau).all(), r
for r in RUNS:                                    # tau < 0: the round-off of the engine (float32), relative to each skewer's peak
    t = S[r].tau
    neg = t < 0
    if neg.any():
        worst = (-t.min(axis=-1) / t.max(axis=-1)).max(axis=-1)       # (n_lines,) the largest |tau_min| / tau_max of a skewer
        print(f"{r}: tau < 0 in {neg.mean():.2%} of the samples (" + ", ".join(
              f"{NAMES[l]}: min {t[j].min():.2g}, |min| / max per skewer <= {worst[j]:.1e}" for j, l in enumerate(LINE_IDS) if t[j].min() < 0) + ")")
print(f"\nreference {REFERENCE}: {REF.sim} snapshot {REF.snap}, z = {Z:.4f}, L = {L:.3f} cMpc, u = {U:.3f} km/s per cMpc (1 km/s = {1 / U:.5f} cMpc), "
      f"v_per = {PERIOD:.2f} km/s; tau finite in every run")
print("sampling of the reference: " + RECS[REFERENCE]["sampling"]["effect"])

for r in RUNS:
    assert np.array_equal(S[r].positions, REF.positions) and np.array_equal(S[r].v, REF.v), r
d = {r: np.abs(S[r].tau - REF.tau) for r in RUNS if r != REFERENCE}
print("the same skewers and pixels in every run; |tau_run - tau_ref| per run, median / 99.9th percentile / max over every "
      "line, skewer and pixel:")
for r, x in d.items():
    print(f"  {r:<13} {np.median(x):.2e} / {np.percentile(x, 99.9):.2e} / {x.max():.2e}")
del d

## Part B — The statistics of every run

The 3D bins are set by the lattice: $|k|$ one bin per mode up to $3.5\,k_f$, steps of $0.4\,k_f$
to 2 cMpc$^{-1}$, then 12 logarithmic bins to the transverse Nyquist frequency $\pi N/L$; the four wedges of
`MU_EDGES` in $\mu = k_\parallel/|k|$. P1D stays on its native modes $k_m = 2\pi m/v_{\rm per}$ for the checks, and is
averaged in 40 logarithmic bins for the figures only. Each line gives the time of each step of a run.

In [ ]:
KF = 2 * np.pi / L
K_NYQ = np.pi * N / L                                            # the transverse Nyquist frequency of the lattice [1/cMpc]
K_EDGES = set_k_edges(KF, k_1=3.5 * KF, k_2=min(2.0, K_NYQ), k_max=K_NYQ, dk_factor=0.4, n_log=12)
K1D_EDGES = np.geomspace(2 * np.pi / PERIOD, np.pi / min(S[r].dv for r in RUNS), 41)   # display bins of P1D [s/km]
P1D_REF = nearest_reference(P1D_DESI_DR1, Z)
K_DESI = float(np.max(P1D_REF["k"]))                             # [s/km]: the largest k of DESI DR1 at the snapshot's z
GRIDS = {}


def grid_for(n, n_pix):
    """The mu wedges of P on an n x n lattice of n_pix pixels (LatticeWedges: the bins of lattice_grid's P_mu alone, the memory of
    512^2 lattices), cached: the same |k| edges and mu wedges for every run."""
    if (n, n_pix) not in GRIDS:
        GRIDS[n, n_pix] = LatticeWedges(n, n_pix, L, k_edges=K_EDGES, mu_edges=MU_EDGES)
    return GRIDS[n, n_pix]


def statistics(s, transmissions=None, n_pix=None, dv=None):
    """run_statistics of one run (or of ``transmissions`` on ``n_pix`` pixels of ``dv``) on its lattice's jackknife blocks
    and bins, T_bar the mean over every pixel (one scale factor: the box), with the time of each step."""
    n_pix = s.v.size if n_pix is None else n_pix
    steps = []
    out = run_statistics(None if transmissions is not None else s.tau, s.dv if dv is None else dv, line_ids=LINE_IDS,
                         labels=block_labels(s.columns, s.n_per_axis, N_BLOCKS), grid=grid_for(s.n_per_axis, n_pix),
                         transmissions=transmissions, a_pix=np.full(n_pix, A_SNAP), log=steps.append)
    return out, steps


STATS, T0 = {}, time.perf_counter()
for r in RUNS:
    t = time.perf_counter()
    STATS[r], steps = statistics(S[r])
    print(f"{r:<13} {time.perf_counter() - t:6.1f} s: " + "; ".join(steps))
CMP = {r: compare_runs(STATS[r], STATS[REFERENCE], k_edges_1d=K1D_EDGES) for r in RUNS if r != REFERENCE}   # display bins
CMP_ALL = {r: compare_runs(STATS[r], STATS[REFERENCE]) for r in RUNS if r != REFERENCE}                  # every mode: the checks
STYLES = run_styles(ORDER, fiducial=FIDUCIAL)
print(f"every run in {time.perf_counter() - T0:.0f} s; P(k, mu) in {len(MU_EDGES) - 1} wedges x {len(K_EDGES) - 1} |k| bins up to "
      f"{K_NYQ:.2f} cMpc^-1; jackknife over {N_BLOCKS} x {N_BLOCKS} squares of {N // N_BLOCKS} x {N // N_BLOCKS} skewers")

CMP_FINEST_ALL = {r: compare_runs(STATS[r], STATS[FINEST]) for r in RUNS if r not in (FINEST, REFERENCE)}

## Part C — The mean transmission

$\bar F$ of every line and of the total, per run, as $100\,(\bar F_{\rm run}/\bar F_{\rm ref} - 1)$ [%] with its
jackknife error (the reference is the line at 0; the grey band is $\pm 0.5\%$, the dotted lines $\pm 0.1\%$ and
$\pm 1\%$; the axis is linear within `LINTHRESH`).

In [ ]:
fig, axs = plot_runs_mean_flux(CMP, STYLES, REFERENCE, field_labels=NAMES, labels=RUN_LABELS, linthresh=LINTHRESH,
                               title=f"{TITLE_STEM}: the mean transmission against {RUN_LABELS[REFERENCE]}")
add_panel_letters(axs)
plt.show()
for r, c in CMP_ALL.items():
    print(f"{r:<13} " + "; ".join(f"{NAMES[f]}: {100 * c['mean_flux']['rel'][j]:+.3g} +- {100 * c['mean_flux']['err'][j]:.2g} %"
                                  for j, f in enumerate(FIELDS)))

## Part D — P1D

Above, $k P_{\rm 1D}/\pi$ of the reference (thick, its jackknife error as a band) and of every run; below, the residual
$100\,(P_{\rm run}/P_{\rm ref} - 1)$ [%] with its jackknife error bars, in the same bands as Part C. The grey bands
lie outside $[2\pi/v_{\rm per}, \pi/dv]$ of the reference; the dashed line is the largest $k$ of the DESI DR1 P1D at
the snapshot's redshift. The residual axes are linear within `LINTHRESH` = 1e-4 %: the differences are tiny.

In [ ]:
fig, axs = plot_runs_p1d(CMP, STYLES, REFERENCE, field_labels=NAMES, labels=RUN_LABELS, linthresh=LINTHRESH,
                         k_limits=(2 * np.pi / PERIOD, np.pi / REF.dv),
                         title=f"{TITLE_STEM}: $P_{{\\rm 1D}}$ against {RUN_LABELS[REFERENCE]}")
for a in axs.ravel():
    a.axvline(K_DESI, color=PALETTE["grey"], ls="--", lw=1.5, zorder=0)
add_panel_letters(axs[0])
plt.show()

## Part E — The wedges of $P(k, \mu)$

One column per wedge: above, $P(k, \mu)$ of the total overflux of the reference (its jackknife error as a band) and of
every run at the reference's bin centres; below, the residual in per cent as in Part D. 

In [ ]:
fig, axs = plot_runs_wedges(CMP, STYLES, REFERENCE, labels=RUN_LABELS, linthresh=LINTHRESH,
                            title=f"{TITLE_STEM}: $P(k, \\mu)$ of the total against {RUN_LABELS[REFERENCE]}")
add_panel_letters(axs[0])
plt.show()

## Part F — The converged range

For every run and statistic, the largest $k$ up to which **every** bin differs from the reference by less than 0.1%,
0.5% and 1% (`k_max_table`: P1D on its native modes, $P(k, \mu)$ on its $|k|$ bins; "$\geq k$" when every bin passes up to
the run's last one; 0 when the first bin already fails). The last columns give the run's last bin and the median
jackknife error of the relative difference: a $k_{\max}$ set by bins whose difference is within that error is limited
by noise, not by the parameter. The figure draws $k_{\max}(0.5\%)$ against the spacing $dx$.


The table is given twice: against the fiducial (the exact crossings) and against the finest spacing.

In [ ]:
ROWS = k_max_table(CMP_ALL, TOLS)
print(f"against {REFERENCE}; P1D on its native modes [s/km], P(k, mu) on its |k| bins [1/cMpc]; '>= k': every bin passes "
      f"up to the run's last one; the DESI DR1 P1D reaches k = {K_DESI:.3f} s/km at z = {P1D_REF['z']}")
print(format_k_max_table(ROWS, TOLS))
fig, axs = plot_k_max_summary(ROWS, X_OF_RUN, xlabel=X_LABEL, xscale=X_SCALE, xticks=X_TICKS, tol=0.005, p1d_k_ref=K_DESI,
                              title=f"{TITLE_STEM}: the largest $k$ converged to 0.5% against {RUN_LABELS[REFERENCE]}")
add_panel_letters(axs)
plt.show()

print(f"\nagainst the finest spacing {FINEST}:")
print(format_k_max_table(k_max_table(CMP_FINEST_ALL, TOLS), TOLS))

**Reading.** With the integrated profile the sampling spacing does not matter. Against the fiducial (the exact crossings),
every spacing from 1 to 0.01 cMpc gives:

* the mean transmission to $3\times10^{-8}$;
* $P(k, \mu)$ of the total to better than 0.1% in every wedge, up to the last bin (7.4 cMpc$^{-1}$);
* $P_\mathrm{1D}$ of Si III and of the total to 0.5% up to the Nyquist frequency (3.1 s/km).

Only $P_\mathrm{1D}$ of $\mathrm{Ly}\alpha$ departs, beyond $k \approx 1.4$–1.6 s/km, 50 times beyond the DESI DR1 range
(0.032 s/km). There it is $10^{-10}$ of its peak and its jackknife error is 10–20%: the float32 round-off of saturated
absorbers ($|\Delta\tau|$ up to $\sim 10^3$ where $\tau \sim 10^8$). The same holds against the finest spacing.

In the sampled route a coarse spacing costs more, not less, because the reach of the dual segments widens: $dx$ = 1 cMpc
took $\sim$5 h for $128^2$ skewers, 0.05 cMpc took 32 min. The crossings route of the fiducial needs no sampling points
(2.5 min).